In [8]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm
from sklearn.metrics import accuracy_score, confusion_matrix, roc_curve, auc,precision_score,recall_score
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, roc_auc_score,f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
import xgboost as xgb
import argparse
import warnings
import mlflow
from mlflow.tracking import MlflowClient
import mlflow.sklearn
from mlflow.models.signature import ModelSignature, infer_signature
from mlflow.types.schema import Schema,ColSpec
import mlflow.xgboost
#import pickle
from pathlib import Path
# ---- Configure warnings and stdout ----
warnings.filterwarnings("ignore", category=UserWarning)
#sys.stdout.reconfigure(encoding='utf-8')
import sys
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")

#### Get Cleaned Dataset with Feature Scaling

In [9]:
# Function: drop rows with non-numeric values
def drop_non_numeric(df_frame):
    cleaned_df = df_frame.copy()
    for col in cleaned_df.columns:
        # Try to convert column to numeric
        cleaned_df[col] = pd.to_numeric(cleaned_df[col], errors='coerce')
    # Drop rows where conversion failed (NaN introduced)
    cleaned_df = cleaned_df.dropna()
    return cleaned_df

def categorize_chol(val):
    if val < 200:
        return "Normal", 0
    elif 200 <= val <= 239:
        return "Medium", 1
    else:
        return "High", 1

# evaluation function
def eval_metrics(actual, pred):
    rmse = np.sqrt(mean_squared_error(actual, pred))
    mae = mean_absolute_error(actual, pred)
    r2 = r2_score(actual, pred)
    return rmse, mae, r2

def get_cleaned_data():
    # Get current working directory    
    cwd = os.getcwd()
    # Specify dataset filename
    filename = "dataset_2190_cholesterol.csv"
    file_path = os.path.join(cwd, filename)
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"❌ Dataset not found at: {file_path}")
    df = pd.read_csv(file_path)
    null_counts = df.isnull().sum()
    #if null_counts.sum() > 0:
    #    print("\n✅ Null values are present in the dataset.")
    #else:
    #    print("\n❌ No null values found in the dataset.")
    # Apply cleaning
    df_clean = drop_non_numeric(df)
    #print("Cleaned shape:", df_clean.shape)
    df_clean[['chol_category_label', 'chol_category_code']] = df_clean['chol'].apply(
        lambda x: pd.Series(categorize_chol(x))
        )
    # Define columns and target
    columns = ['age', 'sex', 'cp', 'trestbps', 'fbs', 'restecg', 'thalach', 'exang',
           'oldpeak', 'slope', 'ca', 'thal', 'num', 'chol_category_code']
    target = "chol_category_code"
    binary_vars = ['sex', 'fbs', 'exang']
    # Separate features and target
    X = df_clean[columns].drop(columns=[target])
    y = df_clean[target]
    
    # Identify numeric columns to scale (exclude binary + target)
    numeric_cols = X.select_dtypes(include=['int64','float64']).columns.tolist()
    numeric_cols = [col for col in numeric_cols if col not in binary_vars]
    
    # Apply StandardScaler
    scaler = StandardScaler()
    X_scaled = X.copy()
    X_scaled[numeric_cols] = scaler.fit_transform(X[numeric_cols])
    
    # Store result in df_scale (features + target)
    df_scale = X_scaled.copy()
    df_scale[target] = y
    
    #print("Scaled dataset preview:")
    #print(df_scale.columns)
    
    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y, test_size=0.2, random_state=42, stratify=y
    )
    #print("\nTrain set shape:", X_train.shape, y_train.shape)
    #print("Test set shape:", X_test.shape, y_test.shape)
    return X_train, X_test, y_train, y_test


In [10]:
def logistic_model_1(X_train, X_test, y_train, y_test):
    # Logistic Regression with elasticnet penalty, saga solver
    # 1. saga + elasticnet + l1_ratio=0.1, C=10
    C_value = 10
    L_1_ratio = 0.1
    Solver = 'saga'
    Panality = 'elasticnet'
    model = LogisticRegression(
    penalty=Panality,
    solver=Solver,
    l1_ratio=L_1_ratio,
    C=C_value,
    max_iter=10000
    )
    model.fit(X_train, y_train)

    # Predictions
    y_pred_prob = model.predict_proba(X_test)[:, 1]  # probability of class 1
    y_pred_class = model.predict(X_test)

    # Metrics
    acc = accuracy_score(y_test, y_pred_class)
    roc_auc = roc_auc_score(y_test, y_pred_prob)
    mse = mean_squared_error(y_test, y_pred_prob)
    mae = mean_absolute_error(y_test, y_pred_prob)

    # Store results in DataFrame
    results_df = pd.DataFrame([{
        "C": C_value,
        "l1_ratio": L_1_ratio,
        'solver': Solver,
        'penalty' : Panality,
        "Accuracy": acc,
        "ROC_AUC": roc_auc,
        "MSE": mse,
        "MAE": mae
    }])
    print("================ Logistice Model ================")
    print(f'Logistice accureacy {acc}')
    return model, results_df


In [11]:
def run_experiment(uri='default_path', experiment_name='default_exp',
                   model_name='logistic_model', run_name='default_run',
                   model_func=None, model_args=None, tags=None):
    # Set tracking directory explicitly
    mlflow.set_tracking_uri(uri)
    print("The set tracking uri is ", mlflow.get_tracking_uri())

    # ✅ Use set_experiment to avoid duplicate errors
    exp = mlflow.set_experiment(experiment_name=experiment_name)
    exp_id = exp.experiment_id
    get_exp = mlflow.get_experiment(exp_id)
    print("Name:", get_exp.name)
    print("Experiment_id:", get_exp.experiment_id)
    print("Artifact Location:", get_exp.artifact_location)
    print("Tags:", get_exp.tags)
    print("Lifecycle_stage:", get_exp.lifecycle_stage)
    print("Creation timestamp:", get_exp.creation_time)

    with mlflow.start_run(experiment_id=exp_id, run_name=run_name):
        if tags:
            mlflow.set_tags(tags)

        # Train and evaluate model
        model, results_df = model_func(**model_args)

        # Log parameters
        for param in ["C", "l1_ratio", "solver", "penalty"]:
            if param in results_df.columns:
                mlflow.log_param(param, results_df.loc[0, param])

        # Log metrics
        for metric in ["Accuracy", "ROC_AUC", "MSE", "MAE"]:
            if metric in results_df.columns:
                mlflow.log_metric(metric, results_df.loc[0, metric])

        # Log model
        mlflow.sklearn.log_model(model, name=model_name, serialization_format="skops")

        # Log artifacts (optional)
        mlflow.log_artifacts("Logistice_Regressions_Models/")

        # Print artifact URI
        artifacts_uri = mlflow.get_artifact_uri()
        print("The artifact path is", artifacts_uri)

    mlflow.end_run()

    # Show last run info
    run = mlflow.last_active_run()
    if run:
        print("Active run id:", run.info.run_id)
        print("Active run name:", run.info.run_name)

        # 🔑 Register model in MLflow Model Registry
        print(' MLflow Model Registry')
        mlflow.register_model(
            model_uri=f"runs:/{run.info.run_id}/{model_name}",
            name="BinaryClassifications"
        )

        # ✅ Dynamically fetch latest version
        client = MlflowClient()
        latest_versions = client.get_latest_versions("BinaryClassifications")
        latest_version = latest_versions[0].version
        print(f"Loading latest model version: {latest_version}")

        relod_model = mlflow.pyfunc.load_model(
            model_uri=f"models:/BinaryClassifications/{latest_version}"
        )
        predicted_qualities = relod_model.predict(model_args['X_test'])
        (rmse, mae, r2) = eval_metrics(model_args['y_test'], predicted_qualities)

        print("  RMSE_test: %s" % rmse)
        print("  MAE_test: %s" % mae)
        print("  R2_test: %s" % r2)


In [12]:
def get_uri_path() -> str:
    """
    Return MLflow tracking URI path based on current working directory.
    Ensures 'mlruns' and '.trash' folders exist.
    """
    cwd = os.getcwd()
    mlruns_path = os.path.join(cwd, "mlruns")
    trash_path = os.path.join(mlruns_path, ".trash")

    # Create both directories if missing
    os.makedirs(mlruns_path, exist_ok=True)
    os.makedirs(trash_path, exist_ok=True)

    return f"file:{mlruns_path}"

In [13]:
def main():
    warnings.filterwarnings("ignore")
    np.random.seed(40)
    # Load data
    X_train, X_test, y_train, y_test = get_cleaned_data()
    # Example usage
    tags = {
    "Work": "WiFi Platform",
    "release.candidate": "MT7981_01",
    "release.version": "1.0.22",
    "dataset": "Cholesterol",
    "experiment.stage": "hyperparameter_tuning",
    "owner": "abhishek",
    "framework": "Binary-Classifications",
    "model.type": "logistic"
    }
    uri_path = get_uri_path()  
    #uri_path='http://:5000'
    run_experiment(uri=uri_path,
                   experiment_name='mlflow_registery_model', 
                   model_name='Binary_Classifications',
                   run_name='Model_runs', 
                   model_func=logistic_model_1,
                   model_args={
                                "X_train": X_train,
                                "X_test": X_test,
                                "y_train": y_train,
                                "y_test": y_test,
        
                                },
                   tags=tags)

In [14]:
if __name__ == "__main__":
    main()

The set tracking uri is  file:d:\ML\MLFlow\MLFlow-8_RegistryModels\mlruns
Name: mlflow_registery_model
Experiment_id: 445544986015039339
Artifact Location: file:d:\ML\MLFlow\MLFlow-8_RegistryModels\mlruns/445544986015039339
Tags: {}
Lifecycle_stage: active
Creation timestamp: 1780504334159
================ Logistice Model ================
Logistice accureacy 0.8333333333333334


Registered model 'BinaryClassifications' already exists. Creating a new version of this model...
2026/06/03 22:07:31 WARNING mlflow.tracking._model_registry.fluent: Run with id 1a69e642f2164a7db29f6145e8073190 has no artifacts at artifact path 'Binary_Classifications', registering model based on models:/m-257681e52ebb4a998b2d3deea1416710 instead


The artifact path is file:d:\ML\MLFlow\MLFlow-8_RegistryModels\mlruns/445544986015039339/1a69e642f2164a7db29f6145e8073190/artifacts
Active run id: 1a69e642f2164a7db29f6145e8073190
Active run name: Model_runs
 MLflow Model Registry
Loading latest model version: 2
  RMSE_test: 0.408248290463863
  MAE_test: 0.16666666666666666
  R2_test: -0.19999999999999996


Created version '2' of model 'BinaryClassifications'.
